# loop

> Tool execution and context-window management.

Backends that return structured tool calls need the same control loop. The loop approves calls, executes them, records results, and asks the model again. It stops when no calls remain or the budget expires.

`ToolLoopMixin` supplies that behaviour. A backend implements two wire operations:

- `_model_step(**kw) -> Resp` returns one completion.
- `_stream_step(**kw)` yields chunk dicts and stores the merged `Resp` in `self._step_res`.

In [ ]:
#| default_exp loop

In [ ]:
#| export
from concurrent.futures import ThreadPoolExecutor
from fastcore.funccall import mk_ns, call_func
from fastcore.all import L, store_attr, patch
from urai.core import (UsageStats, ChatCallback, StreamFormatter, Resp, run_cbs, resp_text,
                       thought, has_tool_call, tool_reminder_)
from urai.tags import dump_raw
from urai.msgs import mk_msg, mk_toolspec, tc_name, mk_tool_res_msg, sum_usage, coerce_tcs
from urai.chat import (Chat, ContextWindowExceededError, is_ctx_error,
                       budget_msg_, cancel_msg_, cancelled_reply_)

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail
from urai.opts import RUNTIMES, ChatOpts, Runtime, register_runtime
from urai.msgs import ToolCall

## Two more stock callbacks

In [ ]:
#| export
class UsageCallback(ChatCallback):
    "Fold each turn's `usage` block, summed across tool rounds, into `chat.use`."
    order = 10
    def after_response(self):
        u = self.chat.turn_res.get('usage') or {}
        self.chat.use += UsageStats(
            u.get('prompt_tokens', 0), u.get('completion_tokens', 0), u.get('total_tokens', 0), 1,
            cached_tokens=u.get('cached_tokens', 0), model=u.get('model'),
            reasoning_tokens=u.get('reasoning_tokens', 0),
            cache_creation_tokens=u.get('cache_creation_tokens', 0), cost=u.get('cost', 0.0))

class ToolReminderCallback(ChatCallback):
    "Ask the model to say what a tool result told it, rather than silently calling the next one."
    order = 30
    def __init__(self, tool_reminder=tool_reminder_): store_attr()
    def before_send(self):
        m = self.chat.turn_msg
        if not self.chat.tools or m is None: return
        if isinstance(m.get('content'), str): m['content'] += self.tool_reminder
        elif isinstance(m.get('content'), list):
            m['content'].append({'type': 'text', 'text': self.tool_reminder})

## The loop

Provider-run calls are recorded but never executed locally. Tool exceptions become result strings that the model can inspect. Approval runs for every proposed call before any tool executes.

In [ ]:
#| export
class ToolLoopMixin:
    "The Python-side tool loop, for backends that get tool calls back as data."
    _ctx_tokens = 0
    _reparse_asked = False

    @property
    def token_count(self):
        "What the window holds after the last turn. `Chat.pct_full` reads it; backends may override."
        return self._ctx_tokens

    def _set_tools(self, tools):
        "Rebuild the wire schemas and the call namespace."
        self.toolspecs = [mk_toolspec(t) for t in L(tools)]
        self.ns = mk_ns([t for t in L(tools) if callable(t)])

    def call_tool(self, tc):
        "Run one approved call against `self.ns`. An exception comes back as an error string."
        if tc.get('server'): return '(run by the provider)'   # the provider's to run, not ours
        try: return call_func(tc_name(tc), (tc.get('function') or {}).get('arguments') or {}, ns=self.ns)
        except Exception as e: return f'{type(e).__name__}: {e}'

    def _approve1(self, tc):
        "Fire `before_tool_calls`, then check `tc` against approval and the tool-call budget."
        self.turn_tc = tc
        for _ in run_cbs(self, 'before_tool_calls'): pass
        over = self._budget_exceeded or (self.max_steps is not None and self._steps >= self.max_steps)
        ok = False if over else (self.approve is None or self.approve(tc))
        if ok: self._steps += 1
        else: self._budget_exceeded = self._budget_exceeded or over
        return ok, (budget_msg_ if over else 'Denied by human operator')

    def _record_tool(self, tc, out):
        "Truncate an over-long result, record it in `hist`, and fire `after_tool_calls`."
        # measured on the string the model will see, not on the object: a tool returning a
        # huge list costs the same context as one returning a huge string
        if self.tool_max_len and len(s := out if isinstance(out, str) else str(out)) > self.tool_max_len:
            out = s[:self.tool_max_len] + ' ...[truncated]'
        self.turn_tc, self.turn_tool_result = tc, out
        self.hist.append(mk_tool_res_msg(tc, out))
        for _ in run_cbs(self, 'after_tool_calls'): pass

    def _run_tools(self, res):
        "Approve every call first, then run the approved ones, in parallel where that was allowed."
        self.hist.append(res)
        tcs = res.get('tool_calls') or []
        oks = [self._approve1(tc) for tc in tcs]
        todo = [tc for tc, (ok, _) in zip(tcs, oks) if ok]
        allowed = self.parallel_tools
        if not isinstance(allowed, bool): allowed = set(allowed or ())
        can_par = len(todo) > 1 and bool(allowed) and all(allowed is True or tc_name(tc) in allowed
                                                          for tc in todo)
        if can_par:
            n = min(len(todo), self.max_parallel_tools or len(todo))
            with ThreadPoolExecutor(max_workers=n, thread_name_prefix='urai-tool') as ex:
                outs = list(ex.map(self.call_tool, todo))
        else: outs = [cancel_msg_ if self.cancelled else self.call_tool(tc) for tc in todo]
        outs = iter(outs)
        for tc, (ok, denial) in zip(tcs, oks): self._record_tool(tc, next(outs) if ok else denial)

    def _refuse_tools(self, res, why=cancel_msg_):
        "Record `res` and a refusal for each call it proposed, without running any."
        self.hist.append(res)
        for tc in res.get('tool_calls') or []: self._record_tool(tc, why)

    def _cut_res(self, text, thought=''):
        "The assistant message for a turn cut mid-stream."
        res = Resp({'role': 'assistant', 'content': text or cancelled_reply_})
        if thought: res['channels'] = {'thought': thought}
        return res

    def _finish_turn(self, res, us, recorded=False):
        "Shared tail: total the turn's usage and record the response."
        if (u := sum_usage(us)): res['usage'] = u
        if us: self._ctx_tokens = (us[-1] or {}).get('total_tokens', self._ctx_tokens)
        self.turn_res = res
        if not recorded: self.hist.append(res)

### Sending

In [ ]:
#| export
reparse_msg_ = ("That reply held a tool call this harness could not read. Send the call again on its own, "
                "as one `<tool_call>` block holding a JSON object with `name` and `arguments`, and nothing else.")

@patch
def _fix_step(self:ToolLoopMixin, res):
    "The ONE place a step's calls are repaired. Returns True when the turn should ask for the call again."
    if (raw := res.pop('raw', None)) and self.toolspecs: dump_raw(self.toolspecs, raw)
    if (tcs := res.get('tool_calls')): coerce_tcs(tcs, self.toolspecs); return False
    if not res.get('tool_parse_failed') or self._reparse_asked: return False
    self._reparse_asked = True          # once per round: a model that cannot re-emit it never will
    self.hist.append(res); self.hist.append(mk_msg(reparse_msg_))
    return True


Every backend hands its step result through one repair point. Coercion happens there, so no backend can forget it, and a reply whose tool call nothing could read asks for the call again instead of ending the turn on an empty message.

In [ ]:
#| export
@patch
def _send(self:ToolLoopMixin, msg, **kw):
    "Send one message, looping until the model stops calling tools or the budget ends the turn."
    self.turn_msg = self.mk_msg(msg)
    self._reparse_asked = False
    self._check_media()
    for _ in run_cbs(self, 'before_send'): pass
    if self.turn_msg is not None: self.hist.append(self.turn_msg)
    us, recorded = [], False
    while True:
        try: res = self._model_step(**kw)
        except Exception as e:
            if not is_ctx_error(self, e): raise
            res = self.recover_context(e, **kw)
        us.append(res.get('usage'))
        if self._fix_step(res): continue            # a call was emitted but unreadable: ask once more
        if not res.get('tool_calls') or self._budget_exceeded: break
        if self.cancelled: self._refuse_tools(res); recorded = True; break
        self._run_tools(res)
        if self.cancelled: recorded = True; break
        if self._budget_exceeded: break
    self._finish_turn(res, us, recorded)
    for _ in run_cbs(self, 'after_response'): pass
    return self.turn_res

### A backend to test against

`_ScriptChat` plays a fixed list of replies, one per model step, and records the tools it ran. Everything the loop does can be tested with it and no model.

In [ ]:
class _ScriptChat(ToolLoopMixin, Chat):
    "A backend whose replies are scripted, for testing the loop itself."
    _runtime, ctx_limit, token_count = 'script', 8192, 0

    def __init__(self, model=None, *, script=(), fail=None, **kw):
        self.script, self.fail, self.steps, self.step_kw = list(script), fail, 0, []
        o = ChatOpts.create(kw.pop('opts', None), **kw)
        self._setup(model, o)
        self._set_tools(self.tools)

    def _next(self, kw):
        self.step_kw.append(kw)
        if self.fail and self.steps == self.fail[0]: self.steps += 1; raise self.fail[1]
        r = self.script[min(self.steps, len(self.script) - 1)]
        self.steps += 1
        return Resp({'role': 'assistant', 'usage': {'total_tokens': 10}, **r})

    def _model_step(self, **kw): return self._next(kw)

    def _stream_step(self, **kw):
        self._step_res = r = self._next(kw)
        if (th := thought(r)): yield {'channels': {'thought': th}}
        if (t := resp_text(r)): yield {'content': [{'type': 'text', 'text': t}]}

register_runtime(Runtime('script', _ScriptChat, ('script-',)))

def add(a: int, b: int) -> int:
    "Add two numbers."
    return a + b

def boom() -> str:
    "Always fails."
    raise ValueError('nope')

def _call(name, **args): return dict(ToolCall(name, args))
def _mk(script, **kw): return Chat('script-1', runtime='script', script=script, tools=[add, boom], **kw)

In [ ]:
c = _mk([{'content': 'no tools needed'}])
test_eq(resp_text(c('hi')), 'no tools needed')
test_eq([m['role'] for m in c.hist], ['user', 'assistant'])
test_eq(c.steps, 1)

In [ ]:
c = _mk([{'content': '', 'tool_calls': [_call('add', a=1, b=2)]}, {'content': 'It is 3.'}])
test_eq(resp_text(c('what is 1+2?')), 'It is 3.')
test_eq([m['role'] for m in c.hist], ['user', 'assistant', 'tool', 'assistant'])
test_eq(c.hist[2]['content'], '3')
test_eq(c.steps, 2)                        # asked again once the result was in

In [ ]:
# the schema decides the argument types, wherever the backend got the call from
c = _mk([{'content': '', 'tool_calls': [_call('add', a='1', b='2')]}, {'content': 'It is 3.'}])
test_eq(resp_text(c('what is 1+2?')), 'It is 3.')
test_eq(c.hist[1]['tool_calls'][0]['function']['arguments'], {'a': 1, 'b': 2})
test_eq(c.hist[2]['content'], '3')          # and so the call actually ran

In [ ]:
# a reply whose call nothing could parse asks once for the call again, rather than ending the turn
c = _mk([{'content': 'Reading it.', 'tool_parse_failed': True},
         {'content': '', 'tool_calls': [_call('add', a=1, b=2)]}, {'content': 'It is 3.'}])
test_eq(resp_text(c('what is 1+2?')), 'It is 3.')
test_eq([m['role'] for m in c.hist], ['user', 'assistant', 'user', 'assistant', 'tool', 'assistant'])
assert 'could not read' in c.hist[2]['content']

# it is asked exactly once: a model that keeps failing ends the turn instead of looping
c = _mk([{'content': 'Reading it.', 'tool_parse_failed': True}])
c('what is 1+2?')
test_eq(c.steps, 2)

In [ ]:
# a tool that raises is reported to the model, not propagated to the caller
c = _mk([{'content': '', 'tool_calls': [_call('boom')]}, {'content': 'That failed.'}])
c('try it')
test_eq(c.hist[2]['content'], 'ValueError: nope')

In [ ]:
# a provider-side call is recorded but never run
c = _mk([{'content': '', 'tool_calls': [dict(ToolCall('web_search', {}, server=True))]},
         {'content': 'Found it.'}])
c('search')
test_eq(c.hist[2]['content'], '(run by the provider)')

### The budget

`max_steps` limits tool calls in one turn. At the limit, the loop records a denial and sends `final_prompt`. The final response can use results already collected.

In [ ]:
call_add = {'content': '', 'tool_calls': [_call('add', a=1, b=2)]}

c = _mk([call_add, call_add, {'content': 'Enough.'}], max_steps=1)
test_eq(resp_text(c('go')), 'Enough.')
test_eq(c._budget_exceeded, True)
test_eq([m['content'] for m in c.hist if m['role'] == 'tool'], ['3', budget_msg_])
test_eq(c.hist[-2]['content'], c.final_prompt)   # the closing round was actually sent

In [ ]:
# approval is asked for every call before any of them runs
seen = []
c = _mk([{'content': '', 'tool_calls': [_call('add', a=1, b=2), _call('add', a=3, b=4)]},
         {'content': 'done'}],
        approve=lambda tc: seen.append(tc.get('function', {}).get('arguments')) or len(seen) == 1)
c('go')
test_eq(len(seen), 2)
test_eq([m['content'] for m in c.hist if m['role'] == 'tool'], ['3', 'Denied by human operator'])

In [ ]:
c = _mk([{'content': '', 'tool_calls': [_call('add', a=1, b=2)]}, {'content': 'ok'}],
        tool_max_len=1)
c('go')
test_eq(c.hist[2]['content'], '3')          # exactly at the limit, so untouched
c = _mk([{'content': '', 'tool_calls': [_call('add', a=100, b=200)]}, {'content': 'ok'}],
        tool_max_len=1)
c('go')
test_eq(c.hist[2]['content'], '3 ...[truncated]')   # an int result is measured as the model sees it

In [ ]:
# turn options reach every model step of the turn
c = _mk([call_add, {'content': 'ok'}])
c('go', temp=0.3)
test_eq(c.step_kw, [{'temp': 0.3}, {'temp': 0.3}])

### Running calls at the same time

Parallel execution is opt-in. `parallel_tools=True` allows every tool. A collection of names allows only those tools, because concurrency safety belongs to the tool rather than the model.

In [ ]:
import time

def slow(n: int) -> int:
    "Sleep briefly, then return n."
    time.sleep(0.05); return n

two = {'content': '', 'tool_calls': [_call('slow', n=1), _call('slow', n=2)]}

c = Chat('script-1', runtime='script', script=[two, {'content': 'ok'}], tools=[slow])
t = time.time(); c('go'); serial = time.time() - t
test_eq([m['content'] for m in c.hist if m['role'] == 'tool'], ['1', '2'])

In [ ]:
c = Chat('script-1', runtime='script', script=[two, {'content': 'ok'}], tools=[slow],
         parallel_tools=True)
t = time.time(); c('go'); par = time.time() - t
test_eq([m['content'] for m in c.hist if m['role'] == 'tool'], ['1', '2'])   # order preserved
assert par < serial

In [ ]:
# a name list allows only those tools; anything else falls back to running them one at a time
c = Chat('script-1', runtime='script', script=[two, {'content': 'ok'}], tools=[slow],
         parallel_tools=['something_else'])
t = time.time(); c('go')
assert time.time() - t >= serial * 0.8

### Cancelling

In [ ]:
c = _mk([call_add, {'content': 'ok'}])
c.cbs = L(c.cbs)
class _CancelAfterFirst(ChatCallback):
    def before_tool_calls(self): self.chat.cancel()
c.add_cb(_CancelAfterFirst)
c('go')
test_eq(c.hist[2]['content'], cancel_msg_)    # approved, but not run
test_eq(c.steps, 1)                           # ...and the model was not asked again

### When the window fills mid-turn

An overflow does not immediately lose the turn. Urai truncates the oldest tool results, rebuilds backend state, and asks once for a final response. A second failure raises `ContextWindowExceededError`.

In [ ]:
#| export
@patch
def recover_context(self:ToolLoopMixin, err, keep_last=4, mx=500, **kw):
    "Window full mid-turn: shrink the oldest tool results, rebuild backend state, and ask for a close."
    idxs = [i for i, m in enumerate(self.hist) if m.get('role') == 'tool']
    for i in (idxs[:-keep_last] if keep_last else idxs):
        c = self.hist[i].get('content')
        if isinstance(c, str) and len(c) > mx:
            self.hist[i]['content'] = c[:mx] + ' ...[truncated to recover context]'
    self._budget_exceeded = self._final_sent = True   # no more tools, and this *is* the final round
    self._recreate_conv()
    self.hist.append(mk_msg(self.final_prompt))
    try: return self._model_step(**kw)
    except Exception:
        raise ContextWindowExceededError(
            f'could not recover after truncating tool results: {err}') from err

In [ ]:
c = _mk([call_add, {'content': 'recovered'}], fail=(1, RuntimeError('n_ctx exceeded')))
c.hist[:] = [{'role': 'tool', 'tool_call_id': f'old{i}', 'name': 'add', 'content': 'x' * 900}
             for i in range(6)]
test_eq(resp_text(c('go')), 'recovered')     # answered, rather than raising
test_eq([m['content'].endswith('...[truncated to recover context]') for m in c.hist[:6]],
        [True, True, True, False, False, False])   # `keep_last` spares the newest results
test_eq(c._budget_exceeded, True)            # no more tools this turn
test_eq(c.hist[-2]['content'], c.final_prompt)

In [ ]:
c = _mk([{'content': 'closed'}])
c.hist[:] = [{'role': 'tool', 'tool_call_id': 'a', 'name': 'add', 'content': 'y' * 900}]
c.recover_context(RuntimeError('full'), keep_last=0)   # keep_last=0 shrinks every result
assert c.hist[0]['content'].endswith('...[truncated to recover context]')
test_eq(len(c.hist[0]['content']), 500 + len(' ...[truncated to recover context]'))

In [ ]:
c = _mk([{'content': 'never reached'}], fail=(0, RuntimeError('n_ctx exceeded')))
c.script = []          # nothing to fall back on, so recovery fails too
test_fail(lambda: c('go'), contains='could not recover')

In [ ]:
# an error that is not a context overflow is still an error
c = _mk([{'content': 'x'}], fail=(0, RuntimeError('connection reset')))
test_fail(lambda: c('go'), contains='connection reset')

### Streaming

The streaming loop follows the same tool rules. It closes an open thinking blockquote before a tool call. It also emits any tool call that the backend returned but did not announce in a chunk.

In [ ]:
#| export
@patch
def _stream(self:ToolLoopMixin, msg, cbs=None, **kw):
    "Stream a turn as markdown chunks. Callbacks in `cbs` live only for this turn."
    added = self.add_cbs(cbs); prev = getattr(self, '_streaming', False); self._streaming = True
    try:
        self.turn_msg = self.mk_msg(msg)
        self._reparse_asked = False
        self._check_media()
        for _ in run_cbs(self, 'before_send'): pass
        if self.turn_msg is not None: self.hist.append(self.turn_msg)
        fmt, us, recorded, cut = StreamFormatter(), [], False, False
        while True:
            said, thought_, announced = [], [], False
            step = self._stream_step(**kw)
            try:
                for o in step:
                    if self.cancelled: cut = True; break
                    said.append(resp_text(o)); thought_.append(thought(o))
                    announced = announced or has_tool_call(o)
                    if (s := self._emit(o, fmt)): yield s
            except Exception as e:
                if not is_ctx_error(self, e): raise
                res = self.recover_context(e, **kw)
                yield self._emit(res, StreamFormatter())
                us.append(res.get('usage')); break
            finally:
                if cut: step.close()
            if fmt._inthink and not self._stream_raw: fmt._inthink = False; yield '\n\n'
            if cut:
                res = self._cut_res(''.join(said), ''.join(thought_))
                self.hist.append(res); recorded = True
                break
            res = self._step_res
            us.append(res.get('usage'))
            if self._fix_step(res): continue        # a call was emitted but unreadable: ask once more
            if not res.get('tool_calls') or self._budget_exceeded: break
            if not announced:
                for tc in res['tool_calls']:
                    o = {'content': [{'type': 'tool_call', 'name': tc_name(tc),
                                      'arguments': (tc.get('function') or {}).get('arguments', {})}]}
                    if (s := self._emit(o, fmt)): yield s
            if self.cancelled: self._refuse_tools(res); recorded = True; break
            self._run_tools(res)
            if self.cancelled: recorded = True; break
            if self._budget_exceeded: break
        self._finish_turn(res, us, recorded)
        yield from run_cbs(self, 'after_response')
        return self.turn_res    # captured by `SaveReturn` and `AsyncChat`'s `.value`
    finally: self._streaming = prev; self.remove_cbs(added)

In [ ]:
c = _mk([{'content': 'streamed plainly'}])
test_eq(''.join(c('hi', stream=True)), 'streamed plainly')

In [ ]:
c = _mk([{'content': 'Let me add.', 'tool_calls': [_call('add', a=1, b=2)]},
         {'content': 'It is 3.'}])
out = ''.join(c('go', stream=True))
test_eq(out, 'Let me add.It is 3.')
test_eq([m['content'] for m in c.hist if m['role'] == 'tool'], ['3'])

In [ ]:
# in raw mode the caller gets the chunk dicts, including the announced call
c = _mk([{'content': 'thinking about it', 'tool_calls': [_call('add', a=1, b=2)]},
         {'content': 'done'}])
kinds = [p['type'] for o in c('go', stream='raw') for p in (o.get('content') or [])]
test_eq(kinds, ['text', 'tool_call', 'text'])

In [ ]:
from fastcore.all import SaveReturn
c = _mk([{'content': 'the end'}])
g = SaveReturn(c('hi', stream=True))
test_eq(list(g), ['the end'])
test_eq(resp_text(g.value), 'the end')       # the final Resp, once drained
test_eq(g.value['usage']['total_tokens'], 10)

## Keeping the window from filling

History eviction works on message groups. A tool-calling assistant message stays with its tool results. Eviction never leaves a call without its result.

In [ ]:
#| export
def msg_groups(hist):
    "Split `hist` into atomic groups: a tool-calling assistant message stays with its results."
    groups, cur = [], []
    for m in hist:
        if m.get('role') == 'tool':
            if cur: cur.append(m); continue
            groups.append([m]); continue         # an orphan result: keep it whole anyway
        if cur: groups.append(cur); cur = []
        if m.get('tool_calls'): cur = [m]
        else: groups.append([m])
    if cur: groups.append(cur)
    return groups

def evict_middle(hist, keep_first=2, keep_last=8):
    "Drop whole groups from the middle of `hist`. Returns `(new_hist, dropped)`."
    groups = msg_groups(hist)
    if len(groups) <= keep_first + keep_last: return list(hist), []
    keep = groups[:keep_first] + (groups[-keep_last:] if keep_last else [])
    drop = groups[keep_first:len(groups) - keep_last] if keep_last else groups[keep_first:]
    return [m for g in keep for m in g], [m for g in drop for m in g]

In [ ]:
hist = [{'role': 'user', 'content': 'q'},
        {'role': 'assistant', 'content': '', 'tool_calls': [_call('add', a=1, b=2)]},
        {'role': 'tool', 'content': '3'},
        {'role': 'assistant', 'content': 'a'}]
test_eq([[m['role'] for m in g] for g in msg_groups(hist)],
        [['user'], ['assistant', 'tool'], ['assistant']])

In [ ]:
plain = [{'role': 'user', 'content': str(i)} for i in range(10)]
kept, dropped = evict_middle(plain, keep_first=2, keep_last=3)
test_eq([m['content'] for m in kept], ['0', '1', '7', '8', '9'])
test_eq(len(dropped), 5)
test_eq(evict_middle(plain[:4], 2, 3), (plain[:4], []))    # short enough, so nothing goes

In [ ]:
# a call and its result are one group, so eviction can never split them
mixed = plain[:2] + hist[1:3] + plain[2:5]
kept, dropped = evict_middle(mixed, keep_first=1, keep_last=1)
test_eq([m['role'] for m in dropped], ['user', 'assistant', 'tool', 'user', 'user'])
test_eq([m['content'] for m in kept], ['0', '4'])

In [ ]:
#| export
_sum_sp = 'Summarize the conversation extract faithfully and briefly. Reply with the summary only.'

class SlidingWindowCallback(ChatCallback):
    "Evict the middle of `hist` before a turn that would overflow. Needs `chat.ctx_limit` set."
    order = 5
    def __init__(self,
                 threshold=0.9,    # evict once the context is this full
                 keep_first=2,     # leading groups to anchor: the task, usually
                 keep_last=8,      # trailing groups to keep: the live thread
                 summarize=False,  # spend one model call replacing the dropped middle with a summary
                 mx=4000):         # characters of dropped conversation to feed the summarizer
        store_attr()

    def _summary(self, dropped):
        "A stand-in for the dropped middle, or None if the model cannot produce one."
        convo = '\n'.join(f"{m.get('role','?')}: {resp_text(m)}" for m in dropped)[:self.mx]
        try: return self.chat.oneshot(
            f'Summarize this earlier part of a conversation:\n\n{convo}', _sum_sp, think=False)
        except Exception: return None

    def before_send(self):
        c = self.chat
        if not getattr(c, 'ctx_limit', None) or c.pct_full < self.threshold: return
        kept, dropped = evict_middle(c.hist, self.keep_first, self.keep_last)
        if not dropped: return
        if self.summarize and (s := self._summary(dropped)):
            # a user/assistant pair, so message alternation survives the splice
            kept = (kept[:self.keep_first] +
                    [{'role': 'user', 'content': f'[Summary of earlier conversation]\n{s}'},
                     {'role': 'assistant', 'content': 'Understood.'}] + kept[self.keep_first:])
        c.hist[:] = kept
        c.evicted = getattr(c, 'evicted', 0) + len(dropped)
        c._recreate_conv()

In [ ]:
c = _mk([{'content': 'ok'}], cbs=[SlidingWindowCallback(threshold=0.5, keep_first=1, keep_last=1)])
c.hist[:] = list(plain)
c.token_count = 0                       # not full yet
c('go')
test_eq(getattr(c, 'evicted', 0), 0)

In [ ]:
c = _mk([{'content': 'ok'}], cbs=[SlidingWindowCallback(threshold=0.5, keep_first=1, keep_last=1)])
c.hist[:] = list(plain)
c.token_count = 8192                    # full
c('go')
test_eq(c.evicted, 8)
test_eq([m['content'] for m in c.hist[:2]], ['0', '9'])

In [ ]:
# with no `ctx_limit` there is nothing to measure against, so it stays out of the way
c = _mk([{'content': 'ok'}], cbs=[SlidingWindowCallback(threshold=0.5, keep_first=1, keep_last=1)])
c.ctx_limit = 0
c.hist[:] = list(plain)
c('go')
test_eq(getattr(c, 'evicted', 0), 0)

In [ ]:
#| hide
del RUNTIMES['script']

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()